In [0]:
from pyspark.sql.functions import col, to_date, row_number
from pyspark.sql.window import Window

silver_schema = dbutils.widgets.get("silver_schema")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")

orders_cleaned = spark.table(f"{silver_schema}.orders_cleaned")




In [0]:
w = Window.partitionBy("order_id").orderBy(col("order_date").desc())

order_with_rn = orders_cleaned.withColumn("rn", row_number().over(w))



In [0]:
orders_dedupe = order_with_rn.filter(col("rn") == 1).drop("rn")


In [0]:
orders_dedupe.write.format("delta").mode("overwrite").saveAsTable(f"{silver_schema}.orders_dedupe")